### Required libraries

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pytz
import seaborn as sns

print(os.getcwd())

In [ ]:
# My swiss knife
import sys

sys.path.append("../../../00_Swiss_knife/")
from df_visualisation import plot_subplots_per_col_dt_X
from period_filter import filter_dataframe

### Import raw dataset

In [ ]:
df = pd.read_csv("../../data/processed/Clean_U6_Indoor_Outdoor_data_20240601-20260215.csv")
# df = pd.read_csv("../../data/processed/single_file.csv")

In [ ]:
print(df.columns)

In [ ]:
# df = df.rename(columns={'Unnamed: 0':'timestamp'})
# df.head()

In [ ]:
# convert timestamp
df = df.copy()
df['timestamp'] = pd.to_datetime(df['timestamp'],utc=True)
df['timestamp'] = df['timestamp'].dt.tz_convert('Europe/Paris')
df.dtypes

In [ ]:
# Define your characters to look for
room = ''
measurement = 'Co2'
sensor_ID = ''
another_char = ''
timestamp = 'timestamp'

# List comprehension for flexible OR condition
cols_selected = [col for col in df.columns if (room in col and measurement in col and sensor_ID in col and another_char in col) or (timestamp in col)]

filtered_df = df[cols_selected]
filtered_df.info()

In [ ]:
# Choice
Co2_list = ["timestamp",
             "EWATCH_3104_Co2_u6_118",
             "WRF06_4_Co2_u6_117",
             "auto_4f3c_Co2_u6_107",
             "NOVOS3_7_Co2_u6_108"]

In [ ]:
df_Co2 = df[Co2_list]
df_Co2.info()

In [ ]:
df_Co2.columns

In [ ]:
# visualisation
visu_Co2 = plot_subplots_per_col_dt_X(x_column="timestamp",df=df_Co2)

### Rename vars 

In [ ]:
df_Co2 = df_Co2.rename(columns={"WRF06_4_Co2_u6_117":"CO2_u6_117",
                                "auto_4f3c_Co2_u6_107":"CO2_u6_107",
                                "NOVOS3_7_Co2_u6_108":"CO2_u6_108",
                                "EWATCH_3104_Co2_u6_118":"CO2_u6_118"})

In [ ]:
df_Co2.info()

In [ ]:
# Recalibration
df_Co2["CO2_u6_108"] = df_Co2["CO2_u6_108"]-580

In [ ]:
# visualisation
visu_Co2 = plot_subplots_per_col_dt_X(x_column="timestamp",df=df_Co2)

In [ ]:
# remove outliers
cols = ["CO2_u6_118", "CO2_u6_117","CO2_u6_107", "CO2_u6_108"]
limits = {
    "CO2_u6_118": (350, 5000),
    "CO2_u6_117": (350, 5000),
    "CO2_u6_107": (350, 5000),
    "CO2_u6_108": (350, 5000)
}

for col in cols:
    min_val, max_val = limits[col]
    mask = (df_Co2[col] < min_val) | (df_Co2[col] > max_val)
    n_outliers = mask.sum()
    print(f"{col}: {n_outliers} valeurs hors limites")
    df_Co2.loc[mask, col] = np.nan

print("Elles ont été supprimées")

In [ ]:
# visualisation
visu_Co2 = plot_subplots_per_col_dt_X(x_column="timestamp",df=df_Co2)